**Code Implementation**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1I5x_Ttpdtlccd0IFHYhUx7shJRwjUY8y?usp=sharing)

In [ ]:
!pip install simpy numpy scipy matplotlib pandas --quiet

In [ ]:
import simpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import math

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── Section 8.3 — Interactive Simulation ────────────────

arrival_box = widgets.Textarea(
    value='0, 4, 5, 9, 10, 18, 25, 26, 32, 33',
    description='Arrival times:',
    layout=widgets.Layout(width='720px', height='70px'),
    style={'description_width': '120px'}
)

service_box = widgets.Textarea(
    value='2, 3, 5, 1, 2, 2, 3, 4, 5, 1',
    description='Service times:',
    layout=widgets.Layout(width='720px', height='70px'),
    style={'description_width': '120px'}
)

servers_slider = widgets.IntSlider(
    value=1,
    min=1,
    max=5,
    step=1,
    description='Servers:',
    continuous_update=False,
    style={'description_width': '120px'},
    layout=widgets.Layout(width='420px')
)

show_details = widgets.Checkbox(
    value=True,
    description='Show calculation details'
)

out = widgets.Output()


# -------------------------------------------------------------------
# Helper functions
# -------------------------------------------------------------------

def parse_list(text):
    return [float(x.strip()) for x in text.split(',') if x.strip() != '']


def run_simpy_queue(arrivals, services, num_servers):
    """
    SimPy queue simulation.
    arrivals = absolute arrival times
    services = service durations
    num_servers = capacity of SimPy Resource
    """

    env = simpy.Environment()
    server = simpy.Resource(env, capacity=num_servers)

    records = []
    event_log = []

    def customer(env, customer_id, arrival_time, service_time):
        # Wait until the customer's absolute arrival time
        yield env.timeout(arrival_time - env.now)

        actual_arrival = env.now
        event_log.append({
            'Time': env.now,
            'Event': f'Customer {customer_id} arrives',
            'Queue Waiting': len(server.queue),
            'In Service': server.count
        })

        with server.request() as request:
            yield request

            start_time = env.now
            waiting_time = start_time - actual_arrival

            event_log.append({
                'Time': env.now,
                'Event': f'Customer {customer_id} begins service',
                'Queue Waiting': len(server.queue),
                'In Service': server.count
            })

            yield env.timeout(service_time)

            finish_time = env.now
            time_in_system = finish_time - actual_arrival

            event_log.append({
                'Time': env.now,
                'Event': f'Customer {customer_id} departs',
                'Queue Waiting': len(server.queue),
                'In Service': server.count
            })

            records.append({
                'Customer': customer_id,
                'Arrival Time': actual_arrival,
                'Service Time': service_time,
                'Server Capacity': num_servers,
                'Execution Begins': start_time,
                'Execution Completes': finish_time,
                'Waiting Time': waiting_time,
                'Time in System': time_in_system
            })

    n = min(len(arrivals), len(services))

    for i in range(n):
        env.process(customer(env, i + 1, arrivals[i], services[i]))

    env.run()

    df = pd.DataFrame(records).sort_values('Customer').reset_index(drop=True)
    event_df = pd.DataFrame(event_log).sort_values('Time').reset_index(drop=True)

    return df, event_df


def queue_length_over_time(df):
    """
    Queue length = customers waiting only.
    Not customers in service.

    This recreates Figure 8.4 style calculation:
    average queue length = area under queue length curve / total time
    """

    event_times = sorted(set(df['Arrival Time']).union(set(df['Execution Begins'])))

    q = 0
    last_t = event_times[0]
    area = 0

    timeline_times = []
    timeline_queues = []
    queue_duration = {}

    for t in event_times:
        duration = t - last_t

        if duration > 0:
            area += q * duration
            queue_duration[q] = queue_duration.get(q, 0) + duration

            timeline_times.append(last_t)
            timeline_queues.append(q)

            timeline_times.append(t)
            timeline_queues.append(q)

        arrivals_now = (df['Arrival Time'] == t).sum()
        starts_now = (df['Execution Begins'] == t).sum()

        # Customers who arrive and immediately start service are not waiting.
        q = q + arrivals_now - starts_now
        q = max(0, q)

        last_t = t

    total_time = df['Execution Completes'].max()

    if last_t < total_time:
        duration = total_time - last_t
        area += q * duration
        queue_duration[q] = queue_duration.get(q, 0) + duration

        timeline_times.append(last_t)
        timeline_queues.append(q)

        timeline_times.append(total_time)
        timeline_queues.append(q)

    return timeline_times, timeline_queues, area, queue_duration


def update(change=None):
    with out:
        clear_output(wait=True)

        try:
            arrivals = parse_list(arrival_box.value)
            services = parse_list(service_box.value)

            if len(arrivals) == 0 or len(services) == 0:
                print("Please enter arrival times and service times.")
                return

            if len(arrivals) != len(services):
                print("Warning: arrival and service lists have different lengths.")
                print("The simulation will use the shorter list.")
                print()

            df, event_df = run_simpy_queue(
                arrivals,
                services,
                servers_slider.value
            )

            total_time = df['Execution Completes'].max()

            times, queues, area_queue, queue_duration = queue_length_over_time(df)

            avg_time_system = df['Time in System'].mean()
            avg_waiting_time = df['Waiting Time'].mean()
            avg_queue_length = area_queue / total_time if total_time > 0 else 0

            total_service_time = df['Service Time'].sum()
            total_possible_service_time = servers_slider.value * total_time
            utilization = total_service_time / total_possible_service_time
            idle_percent = 1 - utilization

            print("=== Section 8.3 — SimPy Queue Simulation ===")
            print()
            print("This version uses SimPy, a discrete-event simulation library.")
            print()
            print(f"Number of customers = {len(df)}")
            print(f"Number of servers = {servers_slider.value}")
            print(f"Total simulation time = {total_time:.2f}")
            print()

            print("=== Customer Event Table ===")
            display(df)

            print()
            print("=== Main Simulation Statistics ===")
            print(f"Average time in system     = {avg_time_system:.3f}")
            print(f"Average waiting time       = {avg_waiting_time:.3f}")
            print(f"Average number in queue    = {avg_queue_length:.3f}")
            print(f"Server utilization         = {100 * utilization:.2f}%")
            print(f"Server idle percentage     = {100 * idle_percent:.2f}%")

            print()
            print("=== Queue Length Duration Breakdown ===")
            for q_len in sorted(queue_duration.keys()):
                print(f"Queue length = {q_len} for {queue_duration[q_len]:.0f} time units")

            if show_details.value:
                print()
                print("=== Calculation Details ===")

                print()
                print("Average time in system:")
                print("  time in system = execution completes - arrival time")
                print(f"  average = {df['Time in System'].sum():.3f} / {len(df)} = {avg_time_system:.3f}")

                print()
                print("Average waiting time:")
                print("  waiting time = execution begins - arrival time")
                print(f"  average = {df['Waiting Time'].sum():.3f} / {len(df)} = {avg_waiting_time:.3f}")

                print()
                print("Average number in queue:")
                print("  average queue length = area under queue-length curve / total time")

                numerator_parts = []
                for q_len in sorted(queue_duration.keys()):
                    duration = queue_duration[q_len]
                    numerator_parts.append(f"{q_len}×{duration:.0f}")

                numerator_string = " + ".join(numerator_parts)

                print(f"  average = ({numerator_string}) / {total_time:.0f}")
                print(f"  average = {area_queue:.0f} / {total_time:.0f} = {avg_queue_length:.3f}")

                print()
                print("Server utilization:")
                print("  utilization = total service time / total possible service time")
                print(f"  utilization = {total_service_time:.3f} / ({servers_slider.value} × {total_time:.3f}) = {utilization:.3f}")

                print()
                print("=== SimPy Event Log ===")
                display(event_df)

            # -------------------------------------------------------------
            # Plot 1: Queue length over time
            # -------------------------------------------------------------
            plt.figure(figsize=(10, 4))
            plt.step(times, queues, where='post')
            plt.title('Figure 8.4 Style — Queue Length Over Time')
            plt.xlabel('Clock Time')
            plt.ylabel('Number Waiting in Queue')
            plt.yticks(range(0, int(max(queues)) + 2))
            plt.grid(True, alpha=0.3)
            plt.show()

            # -------------------------------------------------------------
            # Plot 2: Customer timeline
            # Waiting = dashed line
            # Service = thick solid line
            # -------------------------------------------------------------
            plt.figure(figsize=(10, 5))

            waiting_label_used = False
            service_label_used = False

            for _, row in df.iterrows():
                y = row['Customer']

                if row['Waiting Time'] > 0:
                    plt.plot(
                        [row['Arrival Time'], row['Execution Begins']],
                        [y, y],
                        linestyle='--',
                        linewidth=2,
                        label='Waiting' if not waiting_label_used else ''
                    )
                    waiting_label_used = True

                plt.plot(
                    [row['Execution Begins'], row['Execution Completes']],
                    [y, y],
                    linewidth=4,
                    label='Service' if not service_label_used else ''
                )
                service_label_used = True

                plt.scatter(row['Arrival Time'], y, marker='o')
                plt.scatter(row['Execution Completes'], y, marker='x')

            plt.title('Customer Timeline: Arrival, Waiting, Service, Departure')
            plt.xlabel('Clock Time')
            plt.ylabel('Customer Number')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.show()

            # -------------------------------------------------------------
            # Plot 3: Waiting time per customer
            # -------------------------------------------------------------
            plt.figure(figsize=(10, 4))
            plt.bar(df['Customer'], df['Waiting Time'])
            plt.title('Waiting Time by Customer')
            plt.xlabel('Customer')
            plt.ylabel('Waiting Time')
            plt.grid(True, axis='y', alpha=0.3)
            plt.show()

            # -------------------------------------------------------------
            # Plot 4: Busy / idle summary
            # -------------------------------------------------------------
            busy_idle_df = pd.DataFrame({
                'Category': ['Busy Time', 'Idle Time'],
                'Time': [
                    total_service_time,
                    total_possible_service_time - total_service_time
                ]
            })

            display(busy_idle_df)

            plt.figure(figsize=(7, 4))
            plt.bar(busy_idle_df['Category'], busy_idle_df['Time'])
            plt.title('Total Busy Time vs Idle Time')
            plt.ylabel('Time')
            plt.grid(True, axis='y', alpha=0.3)
            plt.show()

        except Exception as e:
            print("There is an error in the input.")
            print("Make sure you enter comma-separated numbers only.")
            print()
            print(e)


# -------------------------------------------------------------------
# Connect widgets
# -------------------------------------------------------------------

for widget in [arrival_box, service_box, servers_slider, show_details]:
    widget.observe(update, names='value')


display(widgets.VBox([
    widgets.HTML('<h3>Section 8.3 — Interactive Queue Simulation Using SimPy</h3>'),
    widgets.HTML('This uses SimPy instead of manually coding every event from scratch. Change the inputs below and the results update automatically.'),
    arrival_box,
    service_box,
    servers_slider,
    show_details
]))

display(out)

update()

Output()